# Dress the story — identity_story → CatVTON (the RIGHT way to put the jacket on the character)
Redux swamped (it clones a product-shot, it doesn't dress anyone). **CatVTON** is virtual try-on — "put THIS garment
on THIS person" — so the garment axis belongs to a try-on specialist, not the generator. Clean two-stage pipeline:

1. **Stage 1 — `identity_story`** (FLUX.1-dev): face-locked character across scene panels. *(validated GO)*
2. **Stage 2 — CatVTON** (`remyxai/catvton-flux-modular`, FLUX.1-Fill + LoRA): for each panel, inpaint ONLY the
   clothing region (auto agnostic mask) with the leather jacket. The **face is untouched**, so identity survives and
   the garment is actually WORN (not a cloned product shot).

The flat product-shot jacket that swamped Redux is exactly the garment-image format CatVTON wants. Metrics:
ArcFace(dressed vs FACE) — identity preserved through the edit — + CLIP-to-jacket before/after — garment landed.
Two model loads → staged sequentially (free the generator before loading Fill). OPEN-WEIGHT. 80GB A100.

In [ ]:
import subprocess, os
for _ in range(3):
    if subprocess.call(["pip","install","-q","git+https://github.com/huggingface/diffusers.git"])==0: break
!pip install -q transformers accelerate sentencepiece protobuf hf_transfer scikit-image
!pip install -q insightface facexlib onnxruntime-gpu timm einops ftfy opencv-python-headless peft
subprocess.run(["rm","-rf","flux-recipes"])
subprocess.run(["git","clone","-q","-b","main","https://github.com/remyxai/flux-recipes.git"])

In [ ]:
import sys, torch, numpy as np, cv2, gc
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0,"flux-recipes")
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
assert torch.cuda.is_available(); print("GPU:",torch.cuda.get_device_name(0))
from flux_modular import RecipeRunner
runner=RecipeRunner(steps=20)
from transformers import CLIPModel, CLIPProcessor
from PIL import Image, ImageDraw
from IPython.display import display
_clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda").eval(); _cp=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
@torch.no_grad()
def clip_img(a,b):
    px=_cp(images=[a,b],return_tensors="pt").to("cuda"); v=_clip.vision_model(pixel_values=px["pixel_values"]).pooler_output
    e=_clip.visual_projection(v); e=e/e.norm(dim=-1,keepdim=True); return float((e[0]@e[1]).cpu())
def fmt(x): return f"{x:.2f}" if x is not None else "no-face"
def strip(tiles, cell=300, sub=True):
    n=len(tiles); g=Image.new("RGB",(n*cell+(n+1)*8, cell+40),"white"); d=ImageDraw.Draw(g)
    for j,(cap,im,s) in enumerate(tiles):
        x=8+j*(cell+8); g.paste(im.convert("RGB").resize((cell,cell)),(x,4)); d.text((x+4,cell+8),str(cap)[:38],fill="black")
        if sub and s: d.text((x+4,cell+22),str(s)[:38],fill="black")
    display(g)
print("stage-1 runner + CLIP ready")

## Stage 1 — references + `identity_story` panels (FLUX.1-dev)

In [ ]:
BASE={"name":"base","run":"default","inputs":["prompt"],"params":{"guidance":3.5}}
def gen(p, s): return runner.run(BASE, {"prompt":p}, seed=s)
FACE = gen("a studio portrait headshot of a woman with short auburn hair and green eyes, freckles, neutral background, sharp focus", 7)
JACKET = gen("a worn brown leather aviator jacket with a fur collar, product photo on white background, front view", 11)
SCENES=["a half-body portrait reading a map in a lamplit study",
        "a half-body portrait standing on a windswept clifftop at dawn",
        "a half-body portrait at the wheel of a small wooden sailboat"]
C1={"name":"identity_story","run":"batch","inputs":["id_image","scene_prompts"],"params":{"id_weight":1.0,"guidance":3.5}}
panels=runner.run(C1, {"id_image":FACE,"scene_prompts":SCENES}, seed=0)
# grab the InsightFace app (loaded by the identity run) for ArcFace, BEFORE we free the generator
from flux_modular.identity import _PULID
_app=_PULID["enc"].app
def arc_emb(pil):
    fi=_app.get(cv2.cvtColor(np.asarray(pil.convert("RGB")),cv2.COLOR_RGB2BGR))
    if not fi: return None
    fi=sorted(fi,key=lambda x:(x['bbox'][2]-x['bbox'][0])*(x['bbox'][3]-x['bbox'][1]))[-1]
    e=fi['embedding']; return e/(np.linalg.norm(e)+1e-9)
_ref=arc_emb(FACE)
def arc(im):
    e=arc_emb(im); return float(np.dot(e,_ref)) if (e is not None and _ref is not None) else None
strip([("FACE",FACE,""),("JACKET (garment)",JACKET,"")]+[(f"panel{i+1}", panels[i], f"face={fmt(arc(panels[i]))}") for i in range(len(panels))])
print("stage-1 done: face-locked character across scenes (undressed). Garment = the flat product-shot jacket.")

## Free the generator, load CatVTON (FLUX.1-Fill) — two bases don't co-resident cleanly

In [ ]:
del runner; gc.collect(); torch.cuda.empty_cache()
print("freed FLUX.1-dev; mem:", round(torch.cuda.memory_allocated()/1e9,1),"GB")
from diffusers import ModularPipeline
vton = ModularPipeline.from_pretrained("remyxai/catvton-flux-modular", trust_remote_code=True)
vton.load_components(dtype=torch.bfloat16); vton.to("cuda")
print("CatVTON (FLUX.1-Fill + LoRA + segformer masker) loaded")

## Stage 2 — dress each panel in the jacket (auto-mask, face untouched)

In [ ]:
dressed=[]
for i,pan in enumerate(panels):
    g=torch.Generator("cuda").manual_seed(0)
    out=vton(person_image=pan, garment_image=JACKET, height=768, width=576,
             guidance_scale=30.0, num_inference_steps=30, generator=g).images[0]
    dressed.append(out)
    print(f"panel{i+1} dressed")
# before/after per panel: identity kept through the edit? garment landed?
for i in range(len(panels)):
    strip([("garment",JACKET,""),
           (f"panel{i+1} before", panels[i], f"face={fmt(arc(panels[i]))} jkt={clip_img(panels[i],JACKET):.2f}"),
           (f"panel{i+1} DRESSED", dressed[i], f"face={fmt(arc(dressed[i]))} jkt={clip_img(dressed[i],JACKET):.2f}")], cell=320)
print()
print("GO if each DRESSED panel: (1) keeps the FACE (arcface ~ before — CatVTON only edits the torso), and (2) is")
print("wearing the brown fur-collar leather jacket (jkt-CLIP rises before->after + eyeball). That's the worn garment")
print("Redux could not deliver — the garment axis handled by a try-on specialist, composed with the face-lock.")
print("NO-GO if the auto-mask misses (half-body framing too tight / face cropped) or identity drops through the edit.")

## The dressed comic strip

In [ ]:
strip([("FACE",FACE,"")]+[(f"panel{i+1}", dressed[i], f"face={fmt(arc(dressed[i]))}") for i in range(len(dressed))], cell=340)
print("one character, one wardrobe, across the scenes — identity_story (face) + CatVTON (garment), two specialists.")